In [ ]:
import base64, hashlib, os, shutil, subprocess
FILES = {'cascade_diagram.tex': 'JSBGaWd1cmUgMSBvZiB0aGUgTURQSSB2ZXJzaW9uLCBjb21waWxlZCBvbmNlIGludG8gZmlndXJlcy9jYXNjYWRlX2RpYWdyYW0ucGRmIChzZWUgUkVBRE1FLm1kKSBzbyB0aGF0IHRoZQolIGFydGljbGUgaXRzZWxmIG5lZWRzIG5vIFRpa1ogbGlicmFyaWVzLiBTYW1lIGZvbnRzIGFzIHRoZSBhcnRpY2xlOiBUMSArIG1hdGhwYXpvIChQYWxhdGlubyksIDEwIHB0IGJhc2Ugc2l6ZS4KXGRvY3VtZW50Y2xhc3NbYm9yZGVyPTJwdF17c3RhbmRhbG9uZX0KXHVzZXBhY2thZ2VbVDFde2ZvbnRlbmN9Clx1c2VwYWNrYWdle21hdGhwYXpvfQpcdXNlcGFja2FnZXt0aWt6fQpcdXNldGlremxpYnJhcnl7YXJyb3dzLm1ldGEscG9zaXRpb25pbmcsc2hhcGVzLmdlb21ldHJpY30KXGJlZ2lue2RvY3VtZW50fQpcYmVnaW57dGlrenBpY3R1cmV9W2ZvbnQ9XGZvb3Rub3Rlc2l6ZSwgbm9kZSBkaXN0YW5jZT00bW0sCiAgYm94Ly5zdHlsZT17ZHJhdywgcm91bmRlZCBjb3JuZXJzPTJwdCwgYWxpZ249Y2VudGVyLCBtaW5pbXVtIGhlaWdodD04bW0sIGlubmVyIHNlcD0zcHR9LAogIGFyci8uc3R5bGU9ey17U3RlYWx0aFtsZW5ndGg9Mm1tXX19XQpcbm9kZVtib3hdICh4KSB7bWVzc2FnZVxcJHgkfTsKXG5vZGVbYm94LCByaWdodD1vZiB4XSAoZikge2NoYXJhY3RlciBtb2RlbCAkZiRcXChjYWxpYnJhdGVkIFNWTSl9Owpcbm9kZVtiZWxvdz0xbW0gb2YgZiwgZm9udD1cc2NyaXB0c2l6ZSwgYWxpZ249Y2VudGVyXSB7TGl0ZTogJGYkIGFsb25lLFxcdHJhaW5lZCB3aXRob3V0IGF1Z21lbnRhdGlvbn07Clxub2RlW2JveCwgZGlhbW9uZCwgYXNwZWN0PTIuMiwgaW5uZXIgc2VwPTFwdCwgcmlnaHQ9b2YgZl0gKGQpIHskY19mKHgpXGdlXHRhdSQ/fTsKXG5vZGVbYm94LCBhYm92ZSByaWdodD0ybW0gYW5kIDRtbSBvZiBkXSAoeTEpIHskXGhhdHt5fV9mKHgpJH07Clxub2RlW2JveCwgYmVsb3c9N21tIG9mIGRdIChnKSB7RGlzdGlsQkVSVCAkZyR9Owpcbm9kZVtib3gsIHJpZ2h0PTRtbSBvZiBnXSAoeTIpIHskXGhhdHt5fV9nKHgpJH07ClxkcmF3W2Fycl0gKHgpIC0tIChmKTsKXGRyYXdbYXJyXSAoZikgLS0gKGQpOwpcZHJhd1thcnJdIChkKSB8LSBub2RlW3Bvcz0wLjc1LCBhYm92ZV0ge3llc30gKHkxKTsKXGRyYXdbYXJyXSAoZCkgLS0gbm9kZVtyaWdodF0ge25vfSAoZyk7ClxkcmF3W2Fycl0gKGcpIC0tICh5Mik7ClxlbmR7dGlrenBpY3R1cmV9ClxlbmR7ZG9jdW1lbnR9Cg==', 'logo-mdpi.eps': '...'}
SHA = {'cascade_diagram.tex': '467e951602792285e2aeeb230c53f7e44b63873f3a3bd35213bb96a3bf728f8f', 'logo-mdpi.eps': 'f02d31469c6b2666c9de2da8d8731773bce3751d2fa99a27ff4bcf61a106cdcf'}
work = "/tmp/assets"
os.makedirs(work, exist_ok=True)
for name, data in FILES.items():
    raw = base64.b64decode(data)
    digest = hashlib.sha256(raw).hexdigest()
    print(name, digest)
    if digest != SHA[name]:
        raise SystemExit(f"hash mismatch for {name}")
    open(os.path.join(work, name), "wb").write(raw)
subprocess.run("apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y --no-install-recommends "
               "texlive-latex-base texlive-latex-recommended texlive-latex-extra texlive-pictures "
               "texlive-fonts-recommended ghostscript > /tmp/apt.log 2>&1", shell=True, check=True)
subprocess.run(["pdflatex", "-interaction=nonstopmode", "-halt-on-error", "cascade_diagram.tex"], cwd=work, check=True)
subprocess.run(["gs", "-q", "-dSAFER", "-dNOPAUSE", "-dBATCH", "-dEPSCrop", "-sDEVICE=pdfwrite",
                "-sOutputFile=logo-mdpi.pdf", "logo-mdpi.eps"], cwd=work, check=True)
for name in ("cascade_diagram.pdf", "logo-mdpi.pdf", "cascade_diagram.log"):
    shutil.copy(os.path.join(work, name), "/kaggle/working/" + name)
    print("wrote", name, os.path.getsize(os.path.join(work, name)), "bytes")
print(subprocess.run(["pdflatex", "--version"], capture_output=True, text=True).stdout.splitlines()[0])
print(subprocess.run(["gs", "--version"], capture_output=True, text=True).stdout.strip())
